In [1]:
!pip install -q gradio langchain langgraph openai

In [2]:
import os 
from dotenv import load_dotenv

load_dotenv()

#OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if load_dotenv():
    print("OpenAI api key loaded successfully!")
else:
    print("Warning: No openAI API key Found. Please set it in your .env file")

OpenAI api key loaded successfully!


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

In [4]:
# LLM Setup
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 0.7
)

In [5]:
# Define Stete (Memory)
import gradio as gr
from typing import TypedDict, List
# State Definition
class ChatState(TypedDict):
    messages: List

In [6]:
# LangGraph Node
def chatbot(state: ChatState):
    response = llm.invoke(state["messages"])
    return {
        "messages" : state["messages"] + [response]
    }

In [7]:
# Build Graph
graph = StateGraph(ChatState)

graph.add_node("chatbot", chatbot)
graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

In [17]:
# Gradion Function

def chat_with_bot(user_input, history):
    if history is None:
        history = []

    # convert Gradio History -> Langchain Messages
    messages = []
    for msg in history:
        if msg["role"] == "user":
            messages.append(HumanMessage(content = msg["content"]))
        else:
            messages.append(AIMessage(content = msg["content"]))

    # Add new user input
    messages.append(HumanMessage(content=user_input))

    # Add LangGraph
    result = app.invoke({"messages" : messages})
    ai_response = result["messages"][-1].content

    # append in new formats
    history.append({"role": "user", "content" : user_input})
    history.append({"role": "assistant", "content" : ai_response})
    
    return history, history

In [18]:
# Gradio UI

with gr.Blocks() as demo:
    gr.Markdown("## LangGraph Chatbot")

    chatbot_ui = gr.Chatbot()

    msg = gr.Textbox(placeholder = "Type your message here....")
    clear = gr.Button("clear")
    msg.submit(chat_with_bot, [msg, chatbot_ui], [chatbot_ui, chatbot_ui])
    clear.click(lambda: [], None, chatbot_ui)


In [19]:
# Launch

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
